In [ ]:
# ============================================================
# PROJECT 6: DIABETES SEVERITY
# MULTINOMIAL LOGISTIC REGRESSION
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Libraries imported successfully!")


# ------------------------------------------------------------
# 2. UPLOAD DATASET
# ------------------------------------------------------------

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("\nDataset loaded successfully!")
print("File:", file_name)


# ------------------------------------------------------------
# 3. BASIC DATA EXPLORATION
# ------------------------------------------------------------

print("\n========== FIRST 5 ROWS ==========")
display(df.head())

print("\n========== LAST 5 ROWS ==========")
display(df.tail())

print("\n========== DATASET SHAPE ==========")
print(df.shape)

print("\n========== COLUMN NAMES ==========")
print(df.columns.tolist())

print("\n========== DATA TYPES ==========")
print(df.dtypes)

print("\n========== DATASET INFORMATION ==========")
df.info()

print("\n========== STATISTICAL SUMMARY ==========")
display(df.describe(include="all").T)

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

print("\n========== DUPLICATE ROWS ==========")
print(df.duplicated().sum())


# ------------------------------------------------------------
# 4. CLEAN DATA
# ------------------------------------------------------------

# Replace '?' with NaN
df = df.replace("?", np.nan)

# Remove duplicate rows
df = df.drop_duplicates()

print("\nDataset shape after removing duplicates:")
print(df.shape)


# ------------------------------------------------------------
# 5. SET TARGET COLUMN
# ------------------------------------------------------------

# Default target for the common CDC Diabetes Health Indicators dataset
TARGET = "Diabetes_012"

# If your dataset uses another target name, change the line above.
# Examples:
# TARGET = "Severity"
# TARGET = "severity"
# TARGET = "DiabetesSeverity"

# Try to find the target automatically if the default is not present
if TARGET not in df.columns:

    possible_targets = [
        "Severity",
        "severity",
        "DiabetesSeverity",
        "diabetes_severity",
        "Diabetes_012",
        "Outcome"
    ]

    found_target = None

    for col in possible_targets:
        if col in df.columns:
            found_target = col
            break

    if found_target is not None:
        TARGET = found_target

print("\nTarget column:", TARGET)

if TARGET not in df.columns:
    raise ValueError(
        f"Target column '{TARGET}' was not found.\n"
        f"Available columns are:\n{df.columns.tolist()}\n"
        f"Please change the TARGET variable to your actual target column."
    )


# ------------------------------------------------------------
# 6. TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n========== TARGET VALUE COUNTS ==========")
print(df[TARGET].value_counts())

print("\n========== TARGET VALUE PERCENTAGES ==========")
print(df[TARGET].value_counts(normalize=True) * 100)


# Plot target distribution
plt.figure(figsize=(8, 5))

sns.countplot(
    x=df[TARGET].astype(str)
)

plt.title("Diabetes Severity Distribution")
plt.xlabel("Diabetes Severity")
plt.ylabel("Number of Patients")
plt.show()


# ------------------------------------------------------------
# 7. CHECK NUMBER OF CLASSES
# ------------------------------------------------------------

number_of_classes = df[TARGET].nunique()

print("\nNumber of target classes:", number_of_classes)

if number_of_classes < 3:
    raise ValueError(
        "This project requires at least 3 target classes for "
        "multinomial classification. Your target currently has "
        f"{number_of_classes} class(es)."
    )


# ------------------------------------------------------------
# 8. DISPLAY CLASS LABELS
# ------------------------------------------------------------

print("\n========== TARGET CLASSES ==========")

classes = sorted(df[TARGET].dropna().unique())

for i, value in enumerate(classes):
    print(f"Class {i}: {value}")


# ------------------------------------------------------------
# 9. SEPARATE FEATURES AND TARGET
# ------------------------------------------------------------

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)


# ------------------------------------------------------------
# 10. IDENTIFY NUMERIC AND CATEGORICAL FEATURES
# ------------------------------------------------------------

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n========== NUMERIC FEATURES ==========")
print(numeric_features)

print("\n========== CATEGORICAL FEATURES ==========")
print(categorical_features)


# ------------------------------------------------------------
# 11. EXPLORATORY DATA ANALYSIS
# ------------------------------------------------------------

# Numeric feature distributions
if len(numeric_features) > 0:

    X[numeric_features].hist(
        figsize=(16, 12),
        bins=20
    )

    plt.suptitle(
        "Numeric Feature Distributions",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 12. CORRELATION HEATMAP
# ------------------------------------------------------------

if len(numeric_features) > 1:

    plt.figure(figsize=(14, 10))

    correlation_matrix = df[numeric_features].corr()

    sns.heatmap(
        correlation_matrix,
        annot=False,
        cmap="coolwarm",
        linewidths=0.5
    )

    plt.title("Feature Correlation Heatmap")
    plt.show()


# ------------------------------------------------------------
# 13. TARGET VS NUMERIC FEATURES
# ------------------------------------------------------------

for feature in numeric_features[:8]:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        x=df[TARGET].astype(str),
        y=df[feature]
    )

    plt.title(f"{feature} by Diabetes Severity")
    plt.xlabel("Diabetes Severity")
    plt.ylabel(feature)

    plt.show()


# ------------------------------------------------------------
# 14. TARGET VS CATEGORICAL FEATURES
# ------------------------------------------------------------

for feature in categorical_features[:5]:

    plt.figure(figsize=(10, 5))

    sns.countplot(
        data=df,
        x=feature,
        hue=TARGET
    )

    plt.title(f"{feature} by Diabetes Severity")
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 15. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n========== TRAIN-TEST SPLIT ==========")
print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])


# ------------------------------------------------------------
# 16. PREPROCESSING PIPELINES
# ------------------------------------------------------------

# Numeric preprocessing:
# 1. Fill missing values with median
# 2. Standardize values

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


# Categorical preprocessing:
# 1. Fill missing values with most frequent value
# 2. One-hot encode categories

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# 17. MULTINOMIAL LOGISTIC REGRESSION MODEL
# ------------------------------------------------------------

model = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)


# ------------------------------------------------------------
# 18. COMPLETE MACHINE LEARNING PIPELINE
# ------------------------------------------------------------

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)


# ------------------------------------------------------------
# 19. TRAIN MODEL
# ------------------------------------------------------------

print("\nTraining Multinomial Logistic Regression...")

pipeline.fit(
    X_train,
    y_train
)

print("Model training completed!")


# ------------------------------------------------------------
# 20. MAKE PREDICTIONS
# ------------------------------------------------------------

y_pred = pipeline.predict(X_test)

y_proba = pipeline.predict_proba(X_test)


# ------------------------------------------------------------
# 21. MODEL EVALUATION
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n========== MODEL PERFORMANCE ==========")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")


# ------------------------------------------------------------
# 22. MULTICLASS ROC-AUC
# ------------------------------------------------------------

try:

    roc_auc = roc_auc_score(
        y_test,
        y_proba,
        multi_class="ovr",
        average="weighted"
    )

    print(f"ROC-AUC  : {roc_auc:.4f}")

except Exception as e:

    print("ROC-AUC could not be calculated:", e)


# ------------------------------------------------------------
# 23. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 24. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.show()


# ------------------------------------------------------------
# 25. CONFUSION MATRIX WITH CLASS LABELS
# ------------------------------------------------------------

class_labels = pipeline.named_steps["model"].classes_

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_labels,
    yticklabels=class_labels,
    cmap="Blues"
)

plt.title("Diabetes Severity - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


# ------------------------------------------------------------
# 26. GET PROCESSED FEATURE NAMES
# ------------------------------------------------------------

try:

    feature_names = (
        pipeline
        .named_steps["preprocessor"]
        .get_feature_names_out()
    )

    print("\nNumber of processed features:")
    print(len(feature_names))

except Exception as e:

    print("Could not retrieve feature names:", e)


# ------------------------------------------------------------
# 27. MULTINOMIAL LOGISTIC REGRESSION COEFFICIENTS
# ------------------------------------------------------------

try:

    coefficients = (
        pipeline
        .named_steps["model"]
        .coef_
    )

    print("\nCoefficient matrix shape:")
    print(coefficients.shape)

    # Average absolute coefficient across all classes
    importance = np.mean(
        np.abs(coefficients),
        axis=0
    )

    coefficient_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance
    })

    coefficient_df = coefficient_df.sort_values(
        by="Importance",
        ascending=False
    )

    print("\n========== TOP IMPORTANT FEATURES ==========")

    display(
        coefficient_df.head(20)
    )

except Exception as e:

    print(
        "Could not calculate feature importance:",
        e
    )


# ------------------------------------------------------------
# 28. PLOT TOP IMPORTANT FEATURES
# ------------------------------------------------------------

try:

    top_features = coefficient_df.head(15)

    plt.figure(figsize=(10, 7))

    sns.barplot(
        data=top_features,
        x="Importance",
        y="Feature"
    )

    plt.title(
        "Top Features - Multinomial Logistic Regression"
    )

    plt.xlabel("Mean Absolute Coefficient")
    plt.ylabel("Feature")

    plt.tight_layout()
    plt.show()

except Exception as e:

    print(
        "Could not plot feature importance:",
        e
    )


# ------------------------------------------------------------
# 29. CLASS-SPECIFIC COEFFICIENTS
# ------------------------------------------------------------

try:

    for i, class_label in enumerate(class_labels):

        class_coefficients = pd.DataFrame({
            "Feature": feature_names,
            "Coefficient": coefficients[i]
        })

        class_coefficients["AbsoluteCoefficient"] = (
            class_coefficients["Coefficient"].abs()
        )

        class_coefficients = class_coefficients.sort_values(
            by="AbsoluteCoefficient",
            ascending=False
        )

        print(
            f"\n========== TOP FEATURES FOR CLASS {class_label} =========="
        )

        display(
            class_coefficients.head(10)
        )

except Exception as e:

    print(
        "Could not display class-specific coefficients:",
        e
    )


# ------------------------------------------------------------
# 30. ACTUAL VS PREDICTED
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print("\n========== ACTUAL VS PREDICTED ==========")

display(
    comparison.head(20)
)


# ------------------------------------------------------------
# 31. PREDICT A NEW PATIENT
# ------------------------------------------------------------

# Create a new patient using the first row as a template.
# This makes the code work even if your dataset has many columns.

new_patient = X.iloc[[0]].copy()

# Display the sample patient
print("\n========== SAMPLE PATIENT ==========")
display(new_patient)


# Predict diabetes severity
new_prediction = pipeline.predict(
    new_patient
)

new_probability = pipeline.predict_proba(
    new_patient
)

print("\n========== NEW PATIENT PREDICTION ==========")

print(
    "Predicted Diabetes Severity:",
    new_prediction[0]
)

print("\nClass probabilities:")

for class_label, probability in zip(
    class_labels,
    new_probability[0]
):

    print(
        f"Class {class_label}: {probability:.4f}"
    )


# ------------------------------------------------------------
# 32. MODEL SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("PROJECT 6 - FINAL SUMMARY")
print("=" * 60)

print("Dataset shape:", df.shape)
print("Target variable:", TARGET)
print("Number of classes:", number_of_classes)

print("\nModel:")
print("Multinomial Logistic Regression")

print("\nEvaluation Metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

try:
    print(f"ROC-AUC  : {roc_auc:.4f}")
except:
    pass

print("\nThe model predicts multiple diabetes severity classes")
print("using a complete preprocessing and classification pipeline.")

print("=" * 60)

Libraries imported successfully!
